<a href="https://colab.research.google.com/github/DasCadex/Parte3_ingenieria_IA/blob/main/EP3_Observabilidad_Agente_San_Pedro_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISY1010 · EP3 — Observabilidad del agente Colegio San Pedro (EP2)

Instrumenta el agente de la EP2 con logs estructurados, métricas, dashboard y recomendaciones. Patrones tomados solo de `RA3` (IL3.1 a IL3.4).

| Indicador | Dónde |
|---|---|
| IE1/IE2 métricas (precisión, consistencia, latencia, errores, tokens) | secciones 3 y 4 |
| IE3 logs y trazabilidad | secciones 2 y 4 |
| IE4 dashboard (LangSmith + LangChain) | sección 5 |
| IE5 recomendaciones | sección 6 |
| IE8 seguridad y privacidad | secciones 2 y 7 |

Secciones 0 y 1: agente EP2 sin cambios. Requiere los Secrets `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL_SMALL` y `LANGSMITH_API_KEY`.

## 0–1. Agente EP2 (sin cambios)

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q faiss-cpu langchain langchain-classic langchain-community langchain-openai langchain-huggingface sentence-transformers openai python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%pip -q install langsmith pandas

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    for _k in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception as e:
            if _k == "LLM_API_KEY":
                print(f"⚠️ No se pudo leer el Secret LLM_API_KEY: {type(e).__name__}: {e}")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", os.environ.get("LLM_MODEL_SMALL", "openai/gpt-oss-20b"))

if not os.environ.get("LLM_API_KEY"):
    raise RuntimeError("Falta LLM_API_KEY: créala en 🔑 Secrets y activa 'Acceso desde el notebook'.")

import re
import json
import time
from collections import defaultdict
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic.memory import ConversationBufferWindowMemory, ConversationSummaryMemory
from langchain_community.vectorstores import FAISS


def crear_llm(temperatura):
    return ChatOpenAI(
        model=os.environ["LLM_MODEL"],
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=temperatura,
        max_retries=8,
    )


llm = crear_llm(0)
llm_tutor = crear_llm(0.2)
llm_docente = crear_llm(0.7)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Modelos y embeddings configurados.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelos y embeddings configurados.


In [ ]:
# Datos de ejemplo del colegio (reemplazar por los textos, notas y guías reales)
long_text = (
    "Fracciones equivalentes. Dos fracciones son equivalentes cuando representan la misma cantidad. "
    "Para obtener una fracción equivalente se multiplica o se divide el numerador y el denominador por el mismo número. "
    "Ejemplo: 1/2 = 2/4 = 3/6. Para comprobar si dos fracciones son equivalentes se multiplica en cruz: "
    "2/3 y 4/6 son equivalentes porque 2 × 6 = 12 y 3 × 4 = 12. "
    "Comparación de fracciones. Si tienen el mismo denominador, es mayor la de mayor numerador: 5/8 > 3/8. "
    "Si tienen distinto denominador, se amplifican hasta lograr un denominador común: "
    "2/3 = 8/12 y 3/4 = 9/12, por lo tanto 3/4 es mayor que 2/3. "
    "Suma de fracciones. Con el mismo denominador se suman los numeradores: 2/7 + 3/7 = 5/7. "
    "Con distinto denominador primero se busca un denominador común: 1/2 + 1/3 = 3/6 + 2/6 = 5/6. "
    "El ciclo del agua. El agua de mares, ríos y lagos se evapora por el calor del Sol y sube a la atmósfera. "
    "Al enfriarse se condensa y forma nubes. Luego cae como precipitación en forma de lluvia, nieve o granizo. "
    "Sustantivos y adjetivos. El sustantivo nombra personas, animales, lugares o cosas: niña, perro, escuela. "
    "El adjetivo acompaña al sustantivo y dice cómo es: niña alegre, perro pequeño."
)

ejemplo_guia = """GUÍA DE EJERCICIOS - Matemática 5° básico
Tema: suma de fracciones
I. Selección múltiple (1 punto c/u)
1) 1/5 + 2/5 es igual a: a) 3/10 b) 3/5 c) 2/5 d) 1/5
II. Desarrollo (3 puntos c/u)
2) Ana comió 1/3 de una pizza y Luis 1/6. ¿Qué parte comieron entre los dos?
Pauta: 1-b. 2) 2/6 + 1/6 = 3/6 = 1/2.
Rúbrica: 3 pts procedimiento y resultado correctos; 2 pts error de cálculo; 1 pt plantea la operación; 0 pts sin desarrollo."""

CURSO = "5° básico"
HISTORIAL = {"4521": "Matemática 4.1, Lenguaje 5.7, Ciencias 6.0", "3307": "Matemática 6.3, Lenguaje 5.9, Ciencias 6.1"}
PERFILES = {"4521": {"apoyo_atencion": True}, "3307": {"apoyo_atencion": False}}
CURRICULUM = {"Matemática": "Fracciones equivalentes, comparación y suma de fracciones con distinto denominador.",
              "Ciencias Naturales": "Ciclo del agua y cambios de estado.",
              "Lenguaje": "Sustantivos y adjetivos."}
PROGRESO, ALERTAS, BORRADORES = [], [], {}

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len)
chunks = text_splitter.split_text(long_text)
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 4})
print(f"✓ Base de datos vectorial FAISS creada con {len(chunks)} chunks.")

✓ Base de datos vectorial FAISS creada con 3 chunks.


In [ ]:
# --- Memoria de corto plazo: ventana de 10 turnos por sesión (IL2.2) ---
SESIONES = {}

def memoria_sesion(thread_id):
    if thread_id not in SESIONES:
        SESIONES[thread_id] = ConversationBufferWindowMemory(k=10, memory_key="chat_history", return_messages=True)
    return SESIONES[thread_id]


# --- Memoria de largo plazo: resúmenes de sesiones en FAISS, recuperados por similitud semántica ---
memoria_largo = FAISS.from_texts(["inicio de la memoria episódica"], embedding=embeddings, metadatas=[{"codigo": "_"}])

def guardar_episodio(codigo, texto):
    memoria_largo.add_texts([texto], metadatas=[{"codigo": codigo}])

def recordar(codigo, consulta, k=2):
    docs = memoria_largo.similarity_search(consulta, k=k, fetch_k=30, filter={"codigo": codigo})
    return [d.page_content for d in docs]

def cerrar_sesion(thread_id, codigo):
    ventana = SESIONES.pop(thread_id, None)
    if ventana is None or not ventana.chat_memory.messages:
        return "Sin conversación que resumir."
    resumen = ConversationSummaryMemory.from_messages(
        llm=llm, chat_memory=ventana.chat_memory, summarize_step=100).buffer
    guardar_episodio(codigo, f"Resumen de sesión del estudiante {codigo}: {resumen}")
    return resumen


# --- Persistencia en archivos (sin base de datos) ---
def guardar_estado(ruta="estado_agente"):
    os.makedirs(ruta, exist_ok=True)
    with open(f"{ruta}/estado.json", "w", encoding="utf-8") as f:
        json.dump({"PROGRESO": PROGRESO, "ALERTAS": ALERTAS, "BORRADORES": BORRADORES}, f, ensure_ascii=False)
    memoria_largo.save_local(f"{ruta}/faiss_episodios")

def cargar_estado(ruta="estado_agente"):
    global memoria_largo
    with open(f"{ruta}/estado.json", encoding="utf-8") as f:
        datos = json.load(f)
    PROGRESO[:] = datos["PROGRESO"]
    ALERTAS[:] = datos["ALERTAS"]
    BORRADORES.clear()
    BORRADORES.update(datos["BORRADORES"])
    memoria_largo = FAISS.load_local(f"{ruta}/faiss_episodios", embeddings, allow_dangerous_deserialization=True)

In [ ]:
# --- Lógica de decisiones adaptativas y planificación priorizada ---
def errores_frecuentes(curso):
    ultimo = {}
    for p in PROGRESO:
        if p.get("curso", CURSO) == curso and p["resultado"] in ("acierto", "error"):
            ultimo[(p["codigo"], p["tema"])] = p["resultado"]
    por_tema = defaultdict(list)
    for (_, tema), resultado in ultimo.items():
        por_tema[tema].append(resultado == "error")
    return sorted(((t, sum(v) / len(v), len(v)) for t, v in por_tema.items()), key=lambda x: -x[1])


planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un excelente planificador de tareas. Descompón el objetivo en pasos simples y ejecutables, usando solo "
               "estas herramientas: consultar_curriculum, buscar_material, generar_borrador_guia. "
               "Asigna a cada paso una prioridad de 1 a 3 (3 = más alta) según este orden: seguridad y bienestar del "
               "estudiante > precisión académica > eficiencia. Un paso nunca puede depender de otro de menor prioridad. "
               'Responde únicamente con un arreglo JSON, por ejemplo: [{{"paso": "texto", "prioridad": 3}}]'),
    ("human", "Objetivo: {goal}"),
])
planner_chain = planner_prompt | llm_docente

def ordenar_plan(texto):
    try:
        pasos = json.loads(re.search(r"\[.*\]", texto, re.S).group(0))
    except Exception:
        pasos = [{"paso": l.strip(), "prioridad": 1} for l in texto.split("\n") if l.strip()]
    return sorted(pasos, key=lambda p: p.get("prioridad", 1), reverse=True)

def crear_plan(objetivo):
    return ordenar_plan(planner_chain.invoke({"goal": objetivo}).content)

borrador_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente docente. Crea una guía en borrador con exactamente el mismo formato del ejemplo y "
               "refuerza los errores frecuentes del curso."),
    ("human", "Ejemplo:\n{ejemplo}\n\nContenido oficial:\n{contenido}\n\nTema: {tema}\n"
              "Errores frecuentes del curso: {errores}\nObservaciones: {observaciones}"),
])
borrador_chain = borrador_prompt | llm_docente

In [ ]:
# --- Herramientas del agente (consulta, escritura y razonamiento) ---
@tool
def buscar_material(consulta: str) -> str:
    """Busca en el material oficial del colegio los fragmentos relacionados con la consulta."""
    return "\n---\n".join(d.page_content for d in retriever.invoke(consulta)) or "SIN_CONTEXTO"

@tool
def consultar_historial(codigo: str) -> str:
    """Devuelve las notas del estudiante a partir de su código."""
    return HISTORIAL.get(codigo, "Sin historial.")

@tool
def recordar_sesiones(codigo: str, consulta: str) -> str:
    """Recupera, por similitud semántica, resúmenes de sesiones anteriores del estudiante."""
    return " | ".join(recordar(codigo, consulta)) or "Sin sesiones anteriores."

@tool
def consultar_curriculum(asignatura: str) -> str:
    """Devuelve los objetivos de aprendizaje de una asignatura."""
    return CURRICULUM.get(asignatura, "Sin objetivos registrados.")

@tool
def registrar_progreso(codigo: str, tema: str, resultado: str) -> str:
    """Guarda el avance del estudiante. El resultado debe ser 'acierto' o 'error'."""
    if resultado not in ("acierto", "error"):
        return "Resultado inválido: usa 'acierto' o 'error'."
    PROGRESO.append({"codigo": codigo, "curso": CURSO, "tema": tema, "resultado": resultado})
    return "Progreso registrado."

@tool
def evaluar_nivel(codigo: str) -> str:
    """Calcula el nivel de dificultad del estudiante según sus últimos tres resultados."""
    ultimos = [p["resultado"] for p in PROGRESO if p["codigo"] == codigo and p["resultado"] in ("acierto", "error")][-3:]
    if ultimos == ["error"] * 3:
        return "Nivel básico: baja la dificultad y usa ejemplos concretos."
    if ultimos == ["acierto"] * 3:
        return "Nivel avanzado: sube el nivel y propone un desafío."
    return "Nivel intermedio: mantén el nivel."

@tool
def crear_alerta(tema: str, motivo: str) -> str:
    """Envía una alerta al dashboard del profesor."""
    if any(a["tema"] == tema for a in ALERTAS):
        return "La alerta ya existe."
    ALERTAS.append({"tema": tema, "motivo": motivo})
    return f"Alerta creada: {tema}."

@tool
def generar_borrador_guia(tema: str, observaciones: str = "") -> str:
    """Crea una guía en borrador para que el docente la revise y apruebe."""
    contenido = "\n".join(d.page_content for d in retriever.invoke(tema))
    errores = ", ".join(f"{t} ({p:.0%})" for t, p, _ in errores_frecuentes(CURSO) if p > 0) or "sin datos"
    texto = borrador_chain.invoke({"ejemplo": ejemplo_guia, "contenido": contenido, "tema": tema,
                                   "errores": errores, "observaciones": observaciones or "ninguna"}).content
    id_borrador = f"B{len(BORRADORES) + 1}"
    BORRADORES[id_borrador] = {"tema": tema, "texto": texto, "estado": "pendiente_revision"}
    return f"Borrador {id_borrador} (pendiente de revisión docente):\n{texto}"

@tool
def planificar_tarea(solicitud: str) -> str:
    """Divide una solicitud compleja en pasos ordenados por prioridad."""
    return "\n".join(f"{i+1}. [prioridad {p.get('prioridad', 1)}] {p['paso']}" for i, p in enumerate(crear_plan(solicitud)))


tools_tutor = [buscar_material, consultar_historial, evaluar_nivel, registrar_progreso, recordar_sesiones]
tools_docente = [consultar_curriculum, buscar_material, generar_borrador_guia, planificar_tarea]
print("✅ Herramientas definidas.")

✅ Herramientas definidas.


In [ ]:
# --- Agentes (LangChain Agents, como en IL2.1 e IL2.2) ---
def crear_agente(llm_agente, tools, sistema):
    prompt = ChatPromptTemplate.from_messages([
        ("system", sistema),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])
    agent = create_openai_tools_agent(llm_agente, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5, handle_parsing_errors=True)


tutor = crear_agente(llm_tutor, tools_tutor, (
    "Eres un tutor virtual paciente, empático y motivador del Colegio San Pedro Valle Grande, para estudiantes de 5° básico. "
    "Usa frases cortas y un vocabulario amigable. "
    "Responde únicamente con lo que devuelva buscar_material y nunca inventes. "
    "No entregues el resultado directo: guía con pasos, pistas y preguntas. "
    "Sigue las [Directivas] de cada mensaje y usa las [Sesiones anteriores] para dar continuidad. "
    "Cuando el estudiante intente un ejercicio, usa registrar_progreso con 'acierto' o 'error'. "
    "Nunca insultes. Si no sabes la respuesta, o la pregunta no es de las materias del colegio, "
    "anímalo a preguntarle a su profesor."
))

docente = crear_agente(llm_docente, tools_docente, (
    "Eres el Asistente Docente del Colegio San Pedro Valle Grande. Ejecuta el paso que se te indique usando tus herramientas. "
    "Todo material es un borrador y debe ser revisado y aprobado por el docente."
))

FRUSTRACION = ("no entiendo nada", "me rindo", "odio", "no puedo más", "es muy difícil", "soy tonto", "no sirvo")

def chat_tutor(query, codigo, thread_id="sesion-1"):
    memoria = memoria_sesion(thread_id)
    historial = memoria.load_memory_variables({})["chat_history"]
    directivas = [evaluar_nivel.invoke({"codigo": codigo})]
    if any(p in query.lower() for p in FRUSTRACION):
        directivas.append("Hay frustración: da ánimo y una pista más pequeña.")
        PROGRESO.append({"codigo": codigo, "curso": CURSO, "tema": "frustración", "resultado": "frustracion"})
    if PERFILES.get(codigo, {}).get("apoyo_atencion"):
        directivas.append("Usa textos breves, viñetas y un paso a la vez.")
    episodios = " | ".join(recordar(codigo, query)) or "ninguna"
    entrada = (f"[Código del estudiante: {codigo}]\n[Directivas: {' '.join(directivas)}]\n"
               f"[Sesiones anteriores: {episodios}]\n{query}")
    respuesta = tutor.invoke({"input": entrada, "chat_history": historial})["output"]
    memoria.save_context({"input": query}, {"output": respuesta})
    return respuesta

# Plan-and-Execute (IL2.3): los pasos se ordenan por prioridad y cada uno lo ejecuta el agente
ULTIMO_PLAN, ULTIMOS_RESULTADOS = [], []

def chat_docente(solicitud):
    global ULTIMO_PLAN, ULTIMOS_RESULTADOS
    ULTIMO_PLAN = crear_plan(solicitud)
    print("--- PLAN PRIORIZADO ---")
    for i, p in enumerate(ULTIMO_PLAN):
        print(f"{i+1}. [prioridad {p.get('prioridad', 1)}] {p['paso']}")
    resultados = []
    for i, p in enumerate(ULTIMO_PLAN):
        print(f"[Paso {i+1}] Ejecutando: {p['paso']}")
        previo = "\n".join(resultados[-2:]) or "ninguno"
        resultados.append(docente.invoke({"input": f"Solicitud: {solicitud}\nResultados previos: {previo}\n"
                                                    f"Ejecuta solo este paso: {p['paso']}"})["output"])
        time.sleep(1.5)
    ULTIMOS_RESULTADOS = resultados
    return resultados[-1]

In [ ]:
# --- Guardrails (estilo EthicalAgent), monitoreo y enrutador (estilo AgentA/AgentB) ---
class EthicalAgent:
    insultos = ("idiota", "estúpido", "imbécil", "mierda", "puta")

    def answer(self, question):
        if any(p in question.lower() for p in self.insultos):
            return "Conversemos con respeto, por favor. Estoy aquí para ayudarte a estudiar."
        return None

class AgenteMonitoreo:
    def act(self):
        for tema, porcentaje, n in errores_frecuentes(CURSO):
            if porcentaje > 0.40 and n >= 3:
                crear_alerta.invoke({"tema": tema, "motivo": f"{porcentaje:.0%} de {n} estudiantes falla este tema"})
        for codigo in {p["codigo"] for p in PROGRESO}:
            if sum(p["codigo"] == codigo and p["resultado"] == "frustracion" for p in PROGRESO) >= 3:
                crear_alerta.invoke({"tema": f"estudiante {codigo}", "motivo": "frustración recurrente"})

    def informe(self):
        self.act()
        lineas = [f"- {t}: {p:.0%} del curso con error ({n} estudiantes)" for t, p, n in errores_frecuentes(CURSO)]
        alertas = [f"- ⚠ {a['tema']}: {a['motivo']}" for a in ALERTAS]
        return "Errores por tema:\n" + ("\n".join(lineas) or "- sin datos") + "\nAlertas:\n" + ("\n".join(alertas) or "- ninguna")

def anonimizar(texto):
    return re.sub(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b", "[RUT]", texto)

class Enrutador:
    def __init__(self):
        self.guardrail = EthicalAgent()
        self.monitoreo = AgenteMonitoreo()

    def act(self, question, rol="alumno", codigo="4521", thread_id="sesion-1"):
        question = anonimizar(question)
        bloqueo = self.guardrail.answer(question)
        if bloqueo:
            return bloqueo
        if rol == "docente":
            if any(p in question.lower() for p in ("errores", "alertas", "rendimiento", "cuesta")):
                return self.monitoreo.informe()
            return chat_docente(question)
        respuesta = chat_tutor(question, codigo, thread_id)
        self.monitoreo.act()
        return respuesta

enrutador = Enrutador()

## 2. Observabilidad
Logs JSONL con `trace_id`, `span_id`, tipo de span, estado, duración y tokens. No se guarda contenido de usuarios ni claves.

In [ ]:
# --- 2. Capa de observabilidad (IL3.1 / IL3.2): logs JSONL por span ---
import uuid, hashlib, difflib, itertools, unicodedata
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from langsmith import Client, traceable
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage

PROJECT = "ISY1010-EP3-SanPedro"
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except ImportError:
    pass
os.environ.update(LANGSMITH_TRACING="true", LANGSMITH_PROJECT=PROJECT)
assert os.environ.get("LANGSMITH_API_KEY"), "Falta el Secret LANGSMITH_API_KEY"

RUT = re.compile(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b")
RUTA_LOGS = Path("logs/agente_ep3.jsonl")
RUTA_LOGS.parent.mkdir(exist_ok=True)
RUTA_LOGS.write_text("")


def registrar(**campos):
    campos = {"timestamp": datetime.now(timezone.utc).isoformat(), **campos}
    with RUTA_LOGS.open("a", encoding="utf-8") as f:
        f.write(json.dumps(campos, ensure_ascii=False) + "\n")


def categoria_error(exc):  # solo clase y código: nunca el texto de la excepción (IL3.3)
    codigo, nombre = getattr(exc, "status_code", None), type(exc).__name__
    if codigo == 429 or "RateLimit" in nombre:
        return "limite_429"
    if codigo in (401, 403) or "Authentication" in nombre:
        return "autenticacion"
    if "Timeout" in nombre or codigo in (408, 504):
        return "timeout"
    if "Connection" in nombre or codigo in (502, 503):
        return "conexion"
    return "otro"


class Observador(BaseCallbackHandler):
    def __init__(self):
        self.traza, self.abiertos = None, {}

    def _abrir(self, run_id, nombre, tipo, **meta):
        self.abiertos[run_id] = (time.perf_counter(), nombre, tipo, meta)

    def _cerrar(self, run_id, estado, **meta):
        t0, nombre, tipo, previo = self.abiertos.pop(run_id)
        registrar(trace_id=self.traza, span_id=str(run_id)[:8], parent_span_id=self.traza, nombre=nombre,
                  tipo_span=tipo, estado=estado, duracion_ms=round((time.perf_counter() - t0) * 1000, 1),
                  **previo, **meta)

    def on_chat_model_start(self, serialized, messages, *, run_id, **kw):
        texto = " ".join(str(m.content) for lote in messages for m in lote)
        self._abrir(run_id, "llm", "LLM", pii_en_prompt=bool(RUT.search(texto)), mensajes=sum(map(len, messages)))

    def on_llm_start(self, serialized, prompts, *, run_id, **kw):
        self._abrir(run_id, "llm", "LLM", pii_en_prompt=bool(RUT.search(" ".join(prompts))), mensajes=len(prompts))

    def on_llm_end(self, response, *, run_id, **kw):
        uso = (response.llm_output or {}).get("token_usage") or {}
        if not uso:
            u = getattr(getattr(response.generations[0][0], "message", None), "usage_metadata", None) or {}
            uso = {"prompt_tokens": u.get("input_tokens"), "completion_tokens": u.get("output_tokens"),
                   "total_tokens": u.get("total_tokens")}
        self._cerrar(run_id, "ok", tokens_entrada=uso.get("prompt_tokens"),
                     tokens_salida=uso.get("completion_tokens"), tokens_total=uso.get("total_tokens"))

    def on_llm_error(self, error, *, run_id, **kw):
        self._cerrar(run_id, "error", categoria_error=categoria_error(error), tipo_error=type(error).__name__)

    def on_tool_start(self, serialized, input_str, *, run_id, **kw):
        self._abrir(run_id, kw.get("name") or (serialized or {}).get("name", "tool"), "TOOL", entrada_chars=len(str(input_str)))

    def on_tool_end(self, output, *, run_id, **kw):
        salida = str(getattr(output, "content", output))
        self._cerrar(run_id, "vacio" if salida == "SIN_CONTEXTO" else "ok", salida_chars=len(salida))

    def on_tool_error(self, error, *, run_id, **kw):
        self._cerrar(run_id, "error", categoria_error=categoria_error(error), tipo_error=type(error).__name__)


OBS = Observador()


class Observado:
    def __init__(self, objetivo):
        self.objetivo = objetivo

    def invoke(self, entrada, config=None, **kw):
        return self.objetivo.invoke(entrada, config={**(config or {}), "callbacks": [OBS]}, **kw)


tutor, docente, planner_chain, borrador_chain = map(Observado, (tutor, docente, planner_chain, borrador_chain))


def _entrada(i):  # LangSmith no recibe la pregunta (puede traer RUT)
    return {"caso": i.get("caso"), "rol": i.get("rol", "alumno")}

def _salida(o):
    return {"respuesta_chars": len(o["output"] if isinstance(o, dict) else o)}

@traceable(run_type="chain", name="enrutador.act", process_inputs=_entrada, process_outputs=_salida)
def consultar(caso, pregunta, rol="alumno", codigo="4521", thread_id="ep3"):
    OBS.traza = f"{caso}-{uuid.uuid4().hex[:6]}"
    t0, estado, extra, respuesta = time.perf_counter(), "ok", {}, ""
    try:
        respuesta = enrutador.act(pregunta, rol, codigo, thread_id)
    except Exception as exc:
        estado, extra = "error", {"categoria_error": categoria_error(exc), "tipo_error": type(exc).__name__}
    registrar(trace_id=OBS.traza, span_id=OBS.traza, parent_span_id=None, nombre="enrutador.act", tipo_span="AGENT",
              estado=estado, duracion_ms=round((time.perf_counter() - t0) * 1000, 1), caso=caso, rol=rol,
              estudiante=hashlib.sha1(codigo.encode()).hexdigest()[:8], pregunta_chars=len(pregunta),
              respuesta_chars=len(respuesta), **extra)
    return respuesta


print("✅ Observabilidad lista. Logs en", RUTA_LOGS)


✅ Observabilidad lista. Logs en logs/agente_ep3.jsonl


In [ ]:
# --- Pruebas de software de la capa de observabilidad (sin LLM) ---
OBS.traza, t1, t2 = "T-test", uuid.uuid4(), uuid.uuid4()
OBS.on_tool_start({"name": "buscar_material"}, "fracciones", run_id=t1)
OBS.on_tool_end("SIN_CONTEXTO", run_id=t1)
OBS.on_chat_model_start({}, [[HumanMessage(content="Mi RUT es 12.345.678-5")]], run_id=t2)
OBS.on_llm_error(type("RateLimitError", (Exception,), {})("clave sk-secreta"), run_id=t2)

reg = [json.loads(l) for l in RUTA_LOGS.read_text().splitlines()]
assert reg[0]["nombre"] == "buscar_material" and reg[0]["estado"] == "vacio"
assert reg[1]["estado"] == "error" and reg[1]["categoria_error"] == "limite_429" and reg[1]["pii_en_prompt"] is True
assert "sk-secreta" not in RUTA_LOGS.read_text() and "12.345.678-5" not in RUTA_LOGS.read_text()
assert categoria_error(TimeoutError()) == "timeout"
RUTA_LOGS.write_text("")
print("✅ Pruebas de observabilidad aprobadas.")


✅ Pruebas de observabilidad aprobadas.


## 3. Escenarios de prueba
8 casos con variabilidad de datos; los casos C1, C3 y C6 se repiten 3 veces para medir consistencia. `PAUSA_S` y `reps` se pueden ajustar según la cuota de la API.

In [ ]:
# --- 3. Escenarios con variabilidad: parafraseo, fuera de tema, insulto, RUT, perfiles y rol docente ---
MULT = ["multipl", "cruz", "numerador"]
CASOS = [
    dict(caso="C1", q="¿Cómo sé si 2/3 y 4/6 son equivalentes?", debe=MULT, reps=3),
    dict(caso="C2", q="Tengo 2/3 y 4/6, ¿valen lo mismo?", debe=MULT),
    dict(caso="C3", q="¿Quién ganó el mundial de fútbol?", debe=["profesor"], reps=3),
    dict(caso="C4", q="Mi RUT es 12.345.678-5, ayúdame a sumar 1/2 + 1/3", debe=["fraccion", "denominador", "comun"]),
    dict(caso="C5", q="eres un idiota", debe=["respeto"]),
    dict(caso="C6", q="Explícame cómo comparar 3/4 con 2/3", cod="3307", debe=["denominador", "comun", "12"], reps=3),
    dict(caso="C7", q="¿Qué temas le cuestan más al curso?", rol="docente", debe=["errores por tema"]),
    dict(caso="C8", q="Crea una guía de comparación de fracciones para 5° básico", rol="docente", debe=["borrador", "guia"]),
]
PAUSA_S = 3
RESULTADOS = []
for c in CASOS:
    for r in range(c.get("reps", 1)):
        rid = uuid.uuid4()
        resp = consultar(c["caso"], c["q"], c.get("rol", "alumno"), c.get("cod", "4521"), f"{c['caso']}-r{r}",
                         langsmith_extra=dict(run_id=rid, tags=[c["caso"]], metadata={"caso": c["caso"], "rep": r + 1}))
        RESULTADOS.append(dict(caso=c["caso"], rep=r + 1, trace_id=OBS.traza, run_id=rid, respuesta=resp))
        print(f"{c['caso']} r{r + 1}: {resp[:90]!r}")
        time.sleep(PAUSA_S)




> Entering new AgentExecutor chain...


/tmp/ipykernel_410/4154123083.py:6: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  SESIONES[thread_id] = ConversationBufferWindowMemory(k=10, memory_key="chat_history", return_messages=True)
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)



Invoking: `buscar_material` with `{'consulta': '¿Cómo sé si 2/3 y 4/6 son equivalentes?'}`


Fracciones equivalentes. Dos fracciones son equivalentes cuando representan la misma cantidad. Para obtener una fracción equivalente se multiplica o se divide el numerador y el denominador por el mismo número. Ejemplo: 1/2 = 2/4 = 3/6. Para comprobar si dos fracciones son equivalentes se multiplica en cruz: 2/3 y 4/6 son equivalentes porque 2 × 6 = 12 y 3 × 4 = 12. Comparación de fracciones. Si tienen el mismo denominador, es mayor la de mayor numerador: 5/8 > 3/8. Si tienen distinto
---
de mayor numerador: 5/8 > 3/8. Si tienen distinto denominador, se amplifican hasta lograr un denominador común: 2/3 = 8/12 y 3/4 = 9/12, por lo tanto 3/4 es mayor que 2/3. Suma de fracciones. Con el mismo denominador se suman los numeradores: 2/7 + 3/7 = 5/7. Con distinto denominador primero se busca un denominador común: 1/2 + 1/3 = 3/6 + 2/6 = 5/6. El ciclo del agua. El agua de mares, ríos y lagos se evapora

## 4. Métricas y análisis de logs

In [ ]:
# --- 4. Métricas (IE1/IE2) y análisis de logs (IE3) ---
def norm(t):
    return "".join(c for c in unicodedata.normalize("NFD", t.lower()) if unicodedata.category(c) != "Mn")

def cubierto_ms(g):  # tiempo cubierto por spans, sin contar solapes (un padre incluye a sus hijos)
    total, tope = 0.0, None
    for a, b in sorted(zip(g.timestamp - pd.to_timedelta(g.duracion_ms, unit="ms"), g.timestamp)):
        if tope is None or a > tope:
            total, tope = total + (b - a).total_seconds() * 1000, b
        elif b > tope:
            total, tope = total + (b - tope).total_seconds() * 1000, b
    return total

logs = pd.read_json(RUTA_LOGS, lines=True)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True, format="ISO8601")
for col in ("tokens_total", "pii_en_prompt"):
    if col not in logs:
        logs[col] = pd.NA
raiz, hijos = logs[logs.parent_span_id.isna()].set_index("trace_id"), logs[logs.parent_span_id.notna()]
llm, herr = hijos[hijos.tipo_span == "LLM"], hijos[hijos.tipo_span == "TOOL"]

pt = raiz[["caso", "duracion_ms", "estado"]].copy()
pt["llamadas_llm"] = llm.groupby("trace_id").size().reindex(pt.index).fillna(0).astype(int)
pt["llamadas_tool"] = herr.groupby("trace_id").size().reindex(pt.index).fillna(0).astype(int)
pt["tokens"] = pd.to_numeric(llm.tokens_total, errors="coerce").groupby(llm.trace_id).sum(min_count=1).reindex(pt.index)
pt["errores"] = hijos[hijos.estado == "error"].groupby("trace_id").size().reindex(pt.index).fillna(0).astype(int)
cub = hijos.groupby("trace_id").apply(cubierto_ms, include_groups=False).reindex(pt.index).fillna(0)
pt["no_instrumentado_pct"] = (100 * (pt.duracion_ms - cub).clip(lower=0) / pt.duracion_ms).round(1)

res = pd.DataFrame(RESULTADOS)
esperado = {c["caso"]: c["debe"] for c in CASOS}
res["precision_ok"] = [any(k in norm(r) for k in esperado[c]) for c, r in zip(res.caso, res.respuesta)]
tools_por_traza = herr.groupby("trace_id").nombre.apply(lambda s: tuple(sorted(s))).to_dict()
res["herramientas"] = res.trace_id.map(lambda t: tools_por_traza.get(t, ()))

def consistencia(textos):
    pares = list(itertools.combinations([norm(t) for t in textos], 2))
    return sum(difflib.SequenceMatcher(None, a, b).ratio() for a, b in pares) / len(pares)

cons = res.groupby("caso").respuesta.apply(lambda s: consistencia(list(s)) if len(s) > 1 else float("nan")).dropna()
mismas_herr = res.groupby("caso").herramientas.nunique().loc[cons.index].eq(1)
ok = pt[pt.estado == "ok"]
busq = herr[herr.nombre == "buscar_material"]

kpis = pd.DataFrame([
    ("Latencia p50 por solicitud", ok.duracion_ms.quantile(.5) / 1000, "s"),
    ("Latencia p95 por solicitud", ok.duracion_ms.quantile(.95) / 1000, "s"),
    ("Latencia media por llamada LLM", llm.duracion_ms.mean() / 1000, "s"),
    ("Tasa de error (operaciones)", 100 * (logs.estado == "error").mean(), "%"),
    ("Recuperaciones vacías (RAG)", 100 * (busq.estado == "vacio").mean(), "%"),
    ("Tokens medios por solicitud con LLM", pt.tokens.mean(), "tokens"),
    ("Precisión (palabras clave esperadas)", 100 * res.precision_ok.mean(), "%"),
    ("Consistencia de texto entre repeticiones", 100 * cons.mean(), "%"),
    ("Casos con las mismas herramientas en todas las repeticiones", 100 * mismas_herr.mean(), "%"),
    ("Fuga de RUT hacia el LLM", 100 * llm.pii_en_prompt.astype(float).mean(), "%"),
], columns=["metrica", "valor", "unidad"]).round(2)

display(kpis)
display(pt.sort_values("duracion_ms", ascending=False))
display(hijos.sort_values("duracion_ms", ascending=False)[["trace_id", "nombre", "tipo_span", "estado", "duracion_ms"]].head(10))


,metrica,valor,unidad
0,Latencia p50 por solicitud,6.90,s
1,Latencia p95 por solicitud,28.94,s
2,Latencia media por llamada LLM,4.09,s
3,Tasa de error (operaciones),0.00,%
4,Recuperaciones vacías (RAG),0.00,%
5,Tokens medios por solicitud con LLM,2916.17,tokens
6,Precisión (palabras clave esperadas),71.43,%
7,Consistencia de texto entre repeticiones,39.55,%
8,Casos con las mismas herramientas en todas las...,100.00,%
9,Fuga de RUT hacia el LLM,0.00,%


,caso,duracion_ms,estado,llamadas_llm,llamadas_tool,tokens,errores,no_instrumentado_pct
trace_id,,,,,,,,
C8-473958,C8,55267.9,ok,8,3,9992.0,0,8.3
C3-97c433,C3,14768.5,ok,2,1,1971.0,0,0.4
C3-3aa345,C3,10477.3,ok,2,1,2182.0,0,2.5
C3-a90d54,C3,9930.3,ok,2,1,2929.0,0,0.8
C6-0fc503,C6,9329.7,ok,2,1,2012.0,0,0.7
C4-fc7918,C4,8361.0,ok,2,1,1957.0,0,0.8
C6-07a3e9,C6,8199.0,ok,2,1,1808.0,0,0.8
C2-e819ba,C2,5602.2,ok,2,1,2919.0,0,1.2
C1-b3b0b1,C1,3071.2,ok,2,1,3519.0,0,5.9


,trace_id,nombre,tipo_span,estado,duracion_ms
56,C8-473958,llm,LLM,ok,17567.2
47,C8-473958,llm,LLM,ok,14450.4
22,C3-97c433,llm,LLM,ok,13160.5
26,C3-3aa345,llm,LLM,ok,9781.4
53,C8-473958,llm,LLM,ok,9771.3
18,C3-a90d54,llm,LLM,ok,8975.0
35,C6-0fc503,llm,LLM,ok,8882.5
30,C4-fc7918,llm,LLM,ok,7822.8
39,C6-07a3e9,llm,LLM,ok,7805.0
55,C8-473958,generar_borrador_guia,TOOL,ok,5493.6


## 5. Dashboard en LangSmith
La celda envía `precision` y `consistencia` como *feedback* de cada traza. Luego, en LangSmith (proyecto `ISY1010-EP3-SanPedro`):

1. Pestaña **Monitor**: latencia, tasa de error, tokens y costo, calculados por LangSmith.
2. Crear un dashboard propio con gráficos de `precision` y `consistencia` agrupados por la etiqueta del caso (C1 a C8).
3. Capturar los paneles para el informe. El nombre exacto de los menús puede variar.

Las trazas incluyen los prompts de los agentes; en este laboratorio todos los datos son ficticios y el RUT se anonimiza antes.

In [ ]:
# --- 5. Dashboard en LangSmith (IE4): feedback de calidad + estadísticas del proyecto ---
cliente = Client()
wait_for_all_tracers()
getattr(cliente, "flush", lambda: None)()

for r in res.itertuples():
    cliente.create_feedback(run_id=r.run_id, key="precision", score=float(r.precision_ok))
    if r.caso in cons.index:
        cliente.create_feedback(run_id=r.run_id, key="consistencia", score=float(cons[r.caso]))

stats = cliente.get_run_stats(project_names=[PROJECT], is_root=True)
display(pd.Series({k: v for k, v in stats.items() if not isinstance(v, (dict, list))}))
try:
    print("Dashboard:", cliente.read_project(project_name=PROJECT).url)
except Exception:
    print("Abre LangSmith > Tracing projects >", PROJECT)


/tmp/ipykernel_410/2932699885.py:7: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  cliente.create_feedback(run_id=r.run_id, key="precision", score=float(r.precision_ok))
/tmp/ipykernel_410/2932699885.py:9: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  cliente.create_feedback(run_id=r.run_id, key="consistencia", score=float(cons[r.caso]))


,0
run_count,27
latency_p50,5.556437
latency_p99,19.985356
first_token_p50,2.214278
first_token_p99,12.366409
total_tokens,61429
prompt_tokens,37719
completion_tokens,23710
median_tokens,1978
completion_tokens_p50,632


Dashboard: https://smith.langchain.com/o/7a50d413-2e07-451e-a637-abda9485a895/projects/p/5882f4cb-00ad-42da-9bcd-c03eedf00c30


## 6. Recomendaciones

In [ ]:
# --- 6. Hallazgos y recomendaciones (IE5): cada fila sale de los datos medidos ---
H = []
def hallazgo(traza, titulo, evidencia, recomendacion):
    H.append(dict(traza=traza, hallazgo=titulo, evidencia=evidencia, recomendacion=recomendacion))

if len(hijos):
    s = hijos.sort_values("duracion_ms", ascending=False).iloc[0]
    hallazgo(s.trace_id, f"Cuello de botella: {s.nombre} ({s.tipo_span})",
             f"{s.duracion_ms / 1000:.1f} s de {pt.loc[s.trace_id, 'duracion_ms'] / 1000:.1f} s de la solicitud",
             "Acotar max_tokens y usar un modelo más pequeño en pasos simples" if s.tipo_span == "LLM"
             else "Optimizar o cachear la herramienta")

rep = herr.groupby(["trace_id", "nombre"]).size()
for (traza, nombre), n in rep[rep > 1].head(3).items():
    hallazgo(traza, "Herramienta repetida en una solicitud", f"{nombre}: {n} llamadas", "Reutilizar el resultado dentro de la solicitud")

peor = pt.no_instrumentado_pct.idxmax() if len(pt) else None
if peor and pt.loc[peor, "no_instrumentado_pct"] > 20:
    hallazgo(peor, "Tiempo fuera de LLM y herramientas", f"{pt.loc[peor, 'no_instrumentado_pct']:.0f}% de la solicitud",
             "Reemplazar los time.sleep fijos por espera solo ante HTTP 429; revisar reintentos del cliente (max_retries=8)")

err = logs[logs.estado == "error"]
if len(err):
    hallazgo(err.iloc[0].trace_id, "Errores en operaciones", err.categoria_error.value_counts().to_dict(),
             "Límite de solicitudes, backoff acotado y cola de tareas (IL3.4)")

if len(busq) and (busq.estado == "vacio").any():
    hallazgo(busq[busq.estado == "vacio"].iloc[0].trace_id, "Recuperación vacía en el material oficial",
             f"{(busq.estado == 'vacio').mean():.0%} de las búsquedas",
             "Ampliar el material y derivar al profesor cuando no haya contexto")

bajos = cons[cons < .8]
if len(bajos):
    hallazgo("-", "Respuestas poco consistentes", bajos.round(2).to_dict(), "Bajar la temperatura del agente involucrado y fijar el formato de salida")

falla = res[~res.precision_ok]
if len(falla):
    hallazgo(falla.iloc[0].trace_id, "Casos que no cumplen la precisión esperada", sorted(falla.caso.unique()),
             "Revisar a mano esas respuestas y ajustar prompt o base de conocimiento")

if len(pt) and pt.tokens.max() > 2 * pt.tokens.median():
    t = pt.tokens.idxmax()
    hallazgo(t, "Consumo de tokens atípico", f"{pt.tokens.max():.0f} frente a mediana {pt.tokens.median():.0f}",
             "Reducir k del retriever y la ventana de memoria")

hallazgo("-", "Privacidad", f"fuga de RUT al LLM: {100 * llm.pii_en_prompt.astype(float).mean():.0f}%",
         "Mantener anonimizar() antes de cualquier agente y no registrar contenido de usuarios")

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(H))


,traza,hallazgo,evidencia,recomendacion
0,C8-473958,Cuello de botella: llm (LLM),17.6 s de 55.3 s de la solicitud,Acotar max_tokens y usar un modelo más pequeño en pasos simples
1,C7-9c8966,Tiempo fuera de LLM y herramientas,100% de la solicitud,Reemplazar los time.sleep fijos por espera solo ante HTTP 429; revisar reintentos del cliente (max_retries=8)
2,-,Respuestas poco consistentes,"{'C1': 0.03, 'C6': 0.16}",Bajar la temperatura del agente involucrado y fijar el formato de salida
3,C1-b3b0b1,Casos que no cumplen la precisión esperada,"[C1, C3]",Revisar a mano esas respuestas y ajustar prompt o base de conocimiento
4,C8-473958,Consumo de tokens atípico,9992 frente a mediana 1994,Reducir k del retriever y la ventana de memoria
5,-,Privacidad,fuga de RUT al LLM: 0%,Mantener anonimizar() antes de cualquier agente y no registrar contenido de usuarios


## 7. Seguridad y evidencia

In [ ]:
# --- 7. Verificación de seguridad (IE8) y exportación de evidencia ---
texto_logs = RUTA_LOGS.read_text(encoding="utf-8")
assert os.environ["LLM_API_KEY"] not in texto_logs, "Hay una clave en los logs"
assert not RUT.search(texto_logs), "Hay un RUT en los logs"
assert not set(logs.columns) & {"pregunta", "respuesta", "prompt"}, "Los logs no deben guardar contenido de usuarios"
print("✅ Sin claves, RUT ni contenido de usuarios en los logs.")

Path("docs").mkdir(exist_ok=True)
kpis.to_csv("docs/kpis_agente.csv", index=False)
pd.DataFrame(H).to_csv("docs/hallazgos.csv", index=False)
try:
    from google.colab import files
    for f in ("docs/kpis_agente.csv", "docs/hallazgos.csv", str(RUTA_LOGS)):
        files.download(f)
except ImportError:
    pass


✅ Sin claves, RUT ni contenido de usuarios en los logs.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>